In [5]:
import json
import gzip
import csv
import html
import re
from collections import defaultdict
import ftfy

def is_english(text: str) -> bool:
    """检查文本是否为纯英文ASCII"""
    try:
        text.encode('ascii')
        return True
    except UnicodeEncodeError:
        return False

def clean_text(text: str) -> str | None:
    """清洗文本，返回 None 表示不可用"""
    try:
        # HTML entity
        text = html.unescape(text)
        # HTML tags
        text = re.sub(r"<[^>]+>", " ", text)
        # ftfy 修复
        text = ftfy.fix_text(text)
        # UTF-8 round-trip 丢掉无法解码字符
        text = text.encode("utf-8", errors="ignore").decode("utf-8")
        # 合并空格
        text = re.sub(r"\s+", " ", text).strip()
        if not text or not is_english(text):
            return None
        return text
    except Exception:
        return None

def phase1_select_candidates(review_gz: str, review_threshold: int = 40, candidate_limit: int = 3500):
    """Phase 1: 轻量扫描，只收集 review 数量 ≥ threshold 的 product ID，最多 candidate_limit 个"""
    count = defaultdict(int)
    candidates = []
    with gzip.open(review_gz, 'rt', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            asin = data.get("parent_asin")
            if not asin:
                continue
            count[asin] += 1
            if count[asin] == review_threshold:
                candidates.append(asin)
                if len(candidates) >= candidate_limit:
                    break
    return set(candidates)

def is_english_with_symbols(text: str) -> bool:
    """
    判断文本是否只包含英文字符、数字和常见符号
    """
    pattern = r"^[a-zA-Z0-9\s\.\,\!\?\-\'\"\:\;\(\)\&\%\$\@\#]*$"
    return bool(re.match(pattern, text))


def phase2_filter_description(meta_gz: str, candidate_ids: set, final_product_limit: int = 1000):
    """Phase 2: 过滤 description 并清洗，返回最终 product_id -> description"""
    final_products = {}
    with gzip.open(meta_gz, 'rt', encoding='utf-8') as f:
        for line in f:
            meta = json.loads(line)
            asin = meta.get("parent_asin")
            if asin not in candidate_ids:
                continue
            raw_desc_list = meta.get("description", [])
            if not raw_desc_list:
                continue
            raw_desc = " ".join(raw_desc_list).strip()
            if len(raw_desc) < 30 or not is_english_with_symbols(raw_desc):
                continue
            cleaned_desc = clean_text(raw_desc)
            if not cleaned_desc:
                continue
            final_products[asin] = {
                "product_name": meta.get("title", "").strip(),
                "description": cleaned_desc
            }
            if len(final_products) >= final_product_limit:
                break
        
    print(f"Phase 2 合格的 product 数量: {len(final_products)}")

    return final_products


def phase3_collect_reviews(review_gz: str, final_products: dict, review_per_product: int = 30, min_review_len: int = 100):
    """Phase 3: 收集每个 product 的 review"""
    collected = {asin: [] for asin in final_products.keys()}
    with gzip.open(review_gz, 'rt', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            asin = data.get("parent_asin")
            if asin not in final_products:
                continue
            if len(collected[asin]) >= review_per_product:
                continue
            raw_text = data.get("text", "").strip()
            if len(raw_text) < min_review_len or not is_english(raw_text):
                continue
            cleaned = clean_text(raw_text)
            if not cleaned:
                continue
            collected[asin].append({
                "review": cleaned,
                "rating": data.get("rating")
            })
            # 全局提前终止
            all_full = all(len(r) >= review_per_product for r in collected.values())
            if all_full:
                break
    return collected

def save_to_csv(category_name: str, final_products: dict, collected_reviews: dict, output_csv: str):
    """将数据输出为 CSV"""
    with open(output_csv, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=["category", "product_name", "description", "review", "rating"])
        writer.writeheader()
        for asin, product_info in final_products.items():
            reviews = collected_reviews.get(asin, [])
            for rev in reviews:
                writer.writerow({
                    "category": category_name,
                    "product_name": product_info["product_name"],
                    "description": product_info["description"],
                    "review": rev["review"],
                    "rating": rev["rating"]
                })

def process_category(meta_gz: str, review_gz: str, category_name: str, output_csv: str):
    print(f"[{category_name}] Phase 1: Selecting candidate products...")
    candidates = phase1_select_candidates(review_gz)
    print(f"[{category_name}] Phase 2: Filtering and cleaning descriptions...")
    final_products = phase2_filter_description(meta_gz, candidates)
    print(f"[{category_name}] Phase 3: Collecting reviews...")
    collected_reviews = phase3_collect_reviews(review_gz, final_products)
    print(f"[{category_name}] Saving to CSV...")
    save_to_csv(category_name, final_products, collected_reviews, output_csv)
    print(f"[{category_name}] Done! Output saved to {output_csv}")

In [6]:
process_category(
    category_name="Beauty_and_Personal_Care",
    meta_gz="amzn_data/meta_Beauty_and_Personal_Care.jsonl.gz",
    review_gz="amzn_data/Beauty_and_Personal_Care.jsonl.gz",
    output_csv="amzn_data/beauty_1000_products.csv"
)

process_category(
    category_name="Home_and_Kitchen",
    meta_gz="amzn_data/meta_Home_and_Kitchen.jsonl.gz",
    review_gz="amzn_data/Home_and_Kitchen.jsonl.gz",
    output_csv="amzn_data/home_1000_products.csv"
)

[Beauty_and_Personal_Care] Phase 1: Selecting candidate products...
[Beauty_and_Personal_Care] Phase 2: Filtering and cleaning descriptions...
Phase 2 合格的 product 数量: 1000
[Beauty_and_Personal_Care] Phase 3: Collecting reviews...
[Beauty_and_Personal_Care] Saving to CSV...
[Beauty_and_Personal_Care] Done! Output saved to amzn_data/beauty_1000_products.csv
[Home_and_Kitchen] Phase 1: Selecting candidate products...
[Home_and_Kitchen] Phase 2: Filtering and cleaning descriptions...
Phase 2 合格的 product 数量: 1000
[Home_and_Kitchen] Phase 3: Collecting reviews...
[Home_and_Kitchen] Saving to CSV...
[Home_and_Kitchen] Done! Output saved to amzn_data/home_1000_products.csv
